# M2.3 · Categorical features

_Curriculum · Domain 0 · ML Foundations · Feature engineering & leakage_

**Categories are useful only after you encode their meaning without inventing order, exploding width, or leaking the label.**

We compare one-hot encoding, naive target encoding, and out-of-fold smoothed target encoding on campaign IDs. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder

rng = np.random.default_rng(7)

## Build an ads-like categorical dataset

Each row is one impression. The label is a click, and `campaign_id` is deliberately high-cardinality: many campaigns appear only a few times.

In [ ]:
n = 5000
n_campaigns = 1200
campaign_id = rng.integers(0, n_campaigns, size=n)
device = rng.choice(["desktop", "mobile", "tablet"], size=n, p=[0.35, 0.55, 0.10])

campaign_effect = rng.normal(0.0, 0.9, size=n_campaigns)
device_effect = np.where(device == "mobile", 0.25, 0.0)
device_effect = np.where(device == "tablet", -0.20, device_effect)
logit = -2.7 + campaign_effect[campaign_id] + device_effect
prob = 1.0 / (1.0 + np.exp(-logit))
clicked = (rng.random(n) < prob).astype(int)

df = pd.DataFrame({"campaign_id": campaign_id.astype(str), "device": device, "clicked": clicked})
print(df.head())
print("click rate", round(float(df.clicked.mean()), 4))
print("distinct campaigns", df.campaign_id.nunique())

## Smoothing formula

For a category $c$, use

$$\hat{y}_c = \frac{n_c \bar{y}_c + m \bar{y}}{n_c + m}.$$

The formula is not enough by itself. For training rows, the statistic must be computed out-of-fold so a row's own label cannot enter its feature.

In [ ]:
def smoothed_map(frame, key, target, m):
    global_mean = frame[target].mean()
    stats = frame.groupby(key)[target].agg(["count", "mean"])
    values = (stats["count"] * stats["mean"] + m * global_mean) / (stats["count"] + m)
    return values, global_mean

def apply_map(series, mapping, fallback):
    return series.map(mapping).fillna(fallback).to_numpy()

## Naive in-fold target encoding leaks

This version fits the category statistic on the same rows it transforms. Rare categories can therefore receive a feature value partly built from their own label.

In [ ]:
m = 20
mapping, fallback = smoothed_map(df, "campaign_id", "clicked", m)
in_fold_te = apply_map(df["campaign_id"], mapping, fallback)

corr_in_fold = np.corrcoef(in_fold_te, df["clicked"].to_numpy())[0, 1]
print("in-fold target encoding correlation with label", round(float(corr_in_fold), 3))

## Out-of-fold target encoding removes the self-label path

For each fold, fit the encoder on the other folds and transform only the held-out fold. This keeps the training feature honest.

In [ ]:
oof_te = np.zeros(n)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

for train_idx, hold_idx in skf.split(df, df["clicked"]):
    train_frame = df.iloc[train_idx]
    hold_frame = df.iloc[hold_idx]
    fold_map, fold_fallback = smoothed_map(train_frame, "campaign_id", "clicked", m)
    oof_te[hold_idx] = apply_map(hold_frame["campaign_id"], fold_map, fold_fallback)

corr_oof = np.corrcoef(oof_te, df["clicked"].to_numpy())[0, 1]
print("OOF target encoding correlation with label", round(float(corr_oof), 3))

## The suspicious gap is the leakage signal

In-fold target encoding is allowed to peek at each row's label through its category mean. OOF encoding should remain predictive, but less suspiciously attached to the label.

In [ ]:
assert corr_in_fold > corr_oof + 0.05
assert corr_in_fold > 0.20

print("correlation gap", round(float(corr_in_fold - corr_oof), 3))

## Compare one-hot and OOF target encoding in a small model

One-hot keeps categories separate and can work well, but high cardinality creates many sparse columns. OOF target encoding uses one numeric column for `campaign_id`.

In [ ]:
train_idx, test_idx = train_test_split(np.arange(n), test_size=0.3, random_state=7, stratify=df["clicked"])

pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["campaign_id", "device"])])
X_train_oh = pre.fit_transform(df.iloc[train_idx][["campaign_id", "device"]])
X_test_oh = pre.transform(df.iloc[test_idx][["campaign_id", "device"]])

one_hot_model = LogisticRegression(max_iter=500, solver="liblinear")
one_hot_model.fit(X_train_oh, df.iloc[train_idx]["clicked"])
auc_one_hot = roc_auc_score(df.iloc[test_idx]["clicked"], one_hot_model.predict_proba(X_test_oh)[:, 1])

train_map, train_fallback = smoothed_map(df.iloc[train_idx], "campaign_id", "clicked", m)
test_te = apply_map(df.iloc[test_idx]["campaign_id"], train_map, train_fallback)
X_train_te = pd.get_dummies(df.iloc[train_idx]["device"]).to_numpy()
X_test_te = pd.get_dummies(df.iloc[test_idx]["device"]).reindex(columns=pd.get_dummies(df.iloc[train_idx]["device"]).columns, fill_value=0).to_numpy()
X_train_te = np.column_stack([oof_te[train_idx], X_train_te])
X_test_te = np.column_stack([test_te, X_test_te])

te_model = LogisticRegression(max_iter=500, solver="liblinear")
te_model.fit(X_train_te, df.iloc[train_idx]["clicked"])
auc_te = roc_auc_score(df.iloc[test_idx]["clicked"], te_model.predict_proba(X_test_te)[:, 1])

print("one-hot width", X_train_oh.shape[1])
print("target-encoded width", X_train_te.shape[1])
print("one-hot AUC", round(float(auc_one_hot), 3))
print("OOF target-encoded AUC", round(float(auc_te), 3))

## Practice

Try each in the empty cell below.

1. Change `m` from 20 to 100 and watch rare campaign encodings move toward the global mean.
2. Increase `n_campaigns` and compare one-hot width with the single OOF target-encoded column.
3. Replace `campaign_id` with a top-K plus `other` bucket before one-hot encoding.

In [ ]:
# Your turn:
